<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [2]:
raw_data = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
import numpy as np
import pandas as pd
raw_data.head()


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [4]:
raw_data.loc[raw_data['gsc_data_available']==False, [
 'gsc_impressions',
 'gsc_clicks',
]]=np.nan

raw_data.loc[raw_data['ga4_data_available']==False, [
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec']]=np.nan

raw_data.loc[raw_data['gsc_sum_position']==0, 'gsc_sum_position'] = np.nan
raw_data.loc[raw_data['gsc_avg_position']==0, 'gsc_avg_position'] = np.nan

In [5]:
raw_data['ctr'] = (raw_data['gsc_clicks'] / raw_data['gsc_impressions']) * 100
raw_data['engagement_rate'] = (raw_data['ga4_engaged_sessions'] / raw_data['ga4_sessions']) * 100
raw_data['scroll_rate'] = (raw_data['scroll_events'] / raw_data['ga4_pageviews']) * 100
raw_data['scroll_rate'] = raw_data['scroll_rate'].replace([np.inf, -np.inf], 0)
raw_data['scroll_rate'].isin([np.inf, -np.inf]).sum()
raw_data.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,engagement_rate,scroll_rate
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20.0,0.0,67.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1.0,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125.0,1.0,616.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.8,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7.0,0.0,28.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11.0,0.0,25.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>


In [6]:
meta_data = con.sql(f"""
SELECT content_hash_id, competition_level
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
""").df()
raw_data = raw_data.merge(meta_data, on='content_hash_id', how='left')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
competition_map = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
raw_data['competition_level_encoded'] = raw_data['competition_level'].map(competition_map)

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [8]:
"""
1. ctr

Meaning: click-through rate — share of search impressions that resulted in a click, ×100 scale (matches FlyRank's convention)
Missing handling: NaN when gsc_data_available = False (no GSC data at all for this row) — never filled, since fabricating a rate for a page with no search presence would be dishonest
Available when?: known at the decision moment because it's computed from that day's own GSC log — no future data involved

2. gsc_avg_position

Meaning: average search ranking position for that day
Missing handling: NaN when GSC unavailable, OR when the raw value was 0 (a logically impossible position, confirmed via gsc_sum_position cross-check — corrected to null per FlyRank's own documented convention)
Available when?: same-day GSC log, no future leakage

3. engagement_rate

Meaning: share of GA4 sessions that were "engaged" sessions, ×100 scale
Missing handling: NaN when ga4_data_available = False; verified no inf cases (denominator never 0-with-nonzero-numerator in this slice)
Available when?: same-day GA4 log

4. scroll_rate

Meaning: scroll events per pageview, ×100 scale (can exceed 100, matches FlyRank convention)
Missing handling: NaN when GA4 unavailable; the inf edge case (0 pageviews, nonzero scroll events — a GA4 tracking quirk) deliberately set to 0, your judgment call, since "no views but a scroll fired" behaves like near-zero engagement in SEO terms
Available when?: same-day GA4 log

5. competition_level_encoded

Meaning: ordinal encoding of keyword competition intensity (LOW=0, MEDIUM=1, HIGH=2), explicit dictionary mapping (not alphabetical, to preserve real order)
Missing handling: NaN preserved when no keyword data exists for that content item (per dictionary: blank when no keyword context) — not filled, since there's no safe default competition level to assume
Available when?: this is a static content-level attribute from dim_content, not a daily-changing value — known at any decision moment for that content item, since it doesn't depend on the report date
"""

'\n1. ctr\n\nMeaning: click-through rate — share of search impressions that resulted in a click, ×100 scale (matches FlyRank\'s convention)\nMissing handling: NaN when gsc_data_available = False (no GSC data at all for this row) — never filled, since fabricating a rate for a page with no search presence would be dishonest\nAvailable when?: known at the decision moment because it\'s computed from that day\'s own GSC log — no future data involved\n\n2. gsc_avg_position\n\nMeaning: average search ranking position for that day\nMissing handling: NaN when GSC unavailable, OR when the raw value was 0 (a logically impossible position, confirmed via gsc_sum_position cross-check — corrected to null per FlyRank\'s own documented convention)\nAvailable when?: same-day GSC log, no future leakage\n\n3. engagement_rate\n\nMeaning: share of GA4 sessions that were "engaged" sessions, ×100 scale\nMissing handling: NaN when ga4_data_available = False; verified no inf cases (denominator never 0-with-nonz

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [9]:
raw_feb_imp = con.sql(f"""
SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS feb_impressions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY content_hash_id, client_hash_id
""").df()

raw_mar_imp = con.sql(f"""
SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS mar_impressions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY content_hash_id, client_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
trend_df = raw_feb_imp.merge(raw_mar_imp, on=['content_hash_id', 'client_hash_id'], how='inner')

In [11]:
trend_df.head()

,content_hash_id,client_hash_id,feb_impressions,mar_impressions
0,content_7995404695ee1ffd,client_e547b89c05043229,1012.0,768.0
1,content_1eea820697c3b95a,client_e547b89c05043229,299.0,315.0
2,content_ccbb253f142217c3,client_e547b89c05043229,1598.0,3071.0
3,content_5f58c55cbfee172a,client_e547b89c05043229,514.0,387.0
4,content_6fe390ba3af1e456,client_e547b89c05043229,2931.0,4697.0


In [12]:
trend_df = trend_df[trend_df['feb_impressions'] >= 30]
trend_df = trend_df[trend_df['mar_impressions'] > 0]
trend_df['trend_pct'] = (trend_df['mar_impressions'] - trend_df['feb_impressions']) / trend_df['feb_impressions'] * 100

In [13]:
display(trend_df['trend_pct'].describe(),
trend_df['trend_pct'].quantile([0.9, 0.95, 0.99, 0.999]))

,trend_pct
count,96526.000000
mean,114.603801
std,657.862616
min,-99.906015
25%,-9.705904
50%,30.837239
75%,101.087832
max,112758.928571


,trend_pct
0.900,250.648686
0.950,441.381874
0.990,1391.134812
0.999,7709.001241


In [14]:
trend_df['trend_pct'] = trend_df['trend_pct'].clip(lower=-1391.13, upper=1391.13)
trend_df['needs_refresh'] = (trend_df['trend_pct'] < 0).astype(int)

In [15]:
feature_agg = raw_data.groupby(['content_hash_id', 'client_hash_id'])[
    ['ctr', 'gsc_avg_position', 'engagement_rate', 'scroll_rate', 'competition_level_encoded']
].mean().reset_index()

In [16]:
trend_df = trend_df.merge(feature_agg, on=['content_hash_id', 'client_hash_id'], how='inner')

In [17]:
trend_df.head()

,content_hash_id,client_hash_id,feb_impressions,mar_impressions,trend_pct,needs_refresh,ctr,gsc_avg_position,engagement_rate,scroll_rate,competition_level_encoded
0,content_7995404695ee1ffd,client_e547b89c05043229,1012.0,768.0,-24.110672,1,0.149925,34.573578,<NA>,<NA>,0.0
1,content_1eea820697c3b95a,client_e547b89c05043229,299.0,315.0,5.351171,0,0.000000,12.723708,<NA>,<NA>,0.0
2,content_ccbb253f142217c3,client_e547b89c05043229,1598.0,3071.0,92.177722,0,0.231208,24.332310,0.0,0.0,0.0
3,content_5f58c55cbfee172a,client_e547b89c05043229,514.0,387.0,-24.708171,1,0.000000,15.227316,0.0,0.0,0.0
4,content_6fe390ba3af1e456,client_e547b89c05043229,2931.0,4697.0,60.252474,0,0.106907,43.049300,0.0,0.0,0.0


In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X=trend_df[['ctr', 'gsc_avg_position', 'engagement_rate', 'scroll_rate', 'competition_level_encoded']]
y=trend_df['needs_refresh']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model=RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
y_pred=model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.80      0.76     13316
           1       0.41      0.30      0.35      5990

    accuracy                           0.65     19306
   macro avg       0.56      0.55      0.55     19306
weighted avg       0.62      0.65      0.63     19306



In [21]:
X2=trend_df[['ctr', 'trend_pct', 'engagement_rate', 'scroll_rate', 'competition_level_encoded', 'gsc_avg_position']]
y2=trend_df['needs_refresh']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
model=RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X2_train, y2_train)
y2_pred=model.predict(X2_test)

print(classification_report(y2_test, y2_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13316
           1       1.00      1.00      1.00      5990

    accuracy                           1.00     19306
   macro avg       1.00      1.00      1.00     19306
weighted avg       1.00      1.00      1.00     19306



## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.